# Road Damage Detection — YOLO26s Training

Training and evaluation of YOLO26s on the RDD2022 dataset (38,385 labeled images, 4 damage classes).

## Google Drive Setup
Mount Drive to access the dataset archive and save training results.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

from pathlib import Path

drive_project = Path("/content/drive/MyDrive/road-damage-detection")
drive_project.mkdir(parents=True, exist_ok=True)

print("Project folder ready:", drive_project)

Project folder ready: /content/drive/MyDrive/road-damage-detection


In [ ]:
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/road-damage-detection/main_split.zip")

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print("Size:", round(zip_path.stat().st_size / 1024**3, 2), "GiB")

ZIP exists: True
Size: 9.9 GiB


In [ ]:
import shutil

total, used, free = shutil.disk_usage("/content")

print(f"Available storage: {free / 1024**3:.1f} GiB")

Available storage: 70.1 GiB


## Dataset Preparation
Extract the dataset locally in Colab and verify the train/val/test file counts.

In [ ]:

import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/road-damage-detection/main_split.zip")
extract_path = Path("/content/road-damage-detection/data/processed")

extract_path.mkdir(parents=True, exist_ok=True)

print("Extracting dataset to", extract_path, "...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Extraction completed!")

Extracting dataset to /content/road-damage-detection/data/processed ...
Extraction completed!


Verifying the extraction

In [ ]:
from pathlib import Path

# Base directory where the dataset was extracted
base_dir = Path("/content/road-damage-detection/data/processed/main_split")

# Define the paths to check
paths_to_check = {
    "Images Train": base_dir / "images/train",
    "Images Val": base_dir / "images/val",
    "Images Test": base_dir / "images/test",
    "Labels Train": base_dir / "labels/train",
    "Labels Val": base_dir / "labels/val",
    "Labels Test": base_dir / "labels/test"
}

# Count and print the number of files in each directory
for label, path in paths_to_check.items():
    if path.exists():
        file_count = len([f for f in path.iterdir() if f.is_file()])
        print(f"{label}: {file_count} files")
    else:
        print(f"{label} directory does not exist at {path}")

Images Train: 30704 files
Images Val: 3835 files
Images Test: 3846 files
Labels Train: 30704 files
Labels Val: 3835 files
Labels Test: 3846 files


## Project Setup
Clone the GitHub repository, organize the dataset, and install the required libraries.

In [ ]:
!git clone https://github.com/Aaroh22/road-damage-detection.git /content/road-damage-code

Cloning into '/content/road-damage-code'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 69 (delta 32), reused 50 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 1.46 MiB | 7.20 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [ ]:
from pathlib import Path
import shutil

source = Path("/content/road-damage-detection/data")
destination = Path("/content/road-damage-code/data")

if destination.exists():
    print("Destination already exists. No files moved.")
elif not source.is_dir():
    print("Source folder not found. No files moved.")
else:
    shutil.move(str(source), str(destination))
    print("Dataset moved successfully!")

print("Dataset ready:", (destination / "processed/main_split/images/train").is_dir())

Dataset moved successfully!
Dataset ready: True


In [ ]:
from pathlib import Path

project = Path("/content/road-damage-code")

print("GitHub code:", (project / "src/train.py").exists())
print("Dataset:", (project / "data/processed/main_split/images/train").exists())
print("Data config:", (project / "data.yaml").exists())

GitHub code: True
Dataset: True
Data config: True


In [ ]:
#installing ultralytics
%pip install -q ultralytics pyyaml

In [ ]:
from ultralytics import YOLO
import yaml

print("Ultralytics and PyYAML are working!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics and PyYAML are working!


In [ ]:
import sys

project_path = "/content/road-damage-code"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from src.train import prepare_data_yaml

print("Project imports ready!")

Project imports ready!


## Checkpoint Storage
Link YOLO's output directory to Google Drive to preserve training results.

In [ ]:
from pathlib import Path

runs_dir = Path("/content/drive/MyDrive/road-damage-detection/training_runs")

print("Training folder exists:", runs_dir.is_dir())

Training folder exists: True


In [ ]:
from pathlib import Path

local_runs = Path("/content/road-damage-code/runs")
drive_runs = Path("/content/drive/MyDrive/road-damage-detection/training_runs")

if not drive_runs.is_dir():
    raise FileNotFoundError("Google Drive training folder not found")

if local_runs.is_symlink():
    if local_runs.resolve() != drive_runs.resolve():
        raise RuntimeError("runs already points to a different location")
    print("Shortcut already exists.")

elif local_runs.exists():
    raise FileExistsError("runs already exists. Check its contents first.")

else:
    local_runs.symlink_to(drive_runs, target_is_directory=True)
    print("Shortcut created successfully!")

print("YOLO outputs will go to:", local_runs.resolve())

Shortcut created successfully!
YOLO outputs will go to: /content/drive/MyDrive/road-damage-detection/training_runs


## Environment Verification
Verify CUDA availability, GPU access, and required project files.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU runtime — GPU not required yet.")

CUDA available: True
GPU: NVIDIA L4


In [ ]:
from pathlib import Path

project = Path("/content/road-damage-code")
dataset = project / "data/processed/main_split"

print("GitHub code:", (project / "src/train.py").is_file())
print("Dataset:", (dataset / "images/train").is_dir())
print("Data config:", (project / "data.yaml").is_file())
print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Checkpoint shortcut:", (project / "runs").is_symlink())

GitHub code: True
Dataset: True
Data config: True
Drive mounted: True
Checkpoint shortcut: True


## YOLO26s Smoke Test

Train for 1 epoch on 1% of the dataset to verify the training pipeline.

### Configuration
- Model: Pretrained YOLO26s
- Training subset: 1% (307 images)
- Epochs: 1
- Image size: 640 × 640
- Batch size: 8
- Device: NVIDIA Tesla T4
- Seed: 42


This is a technical validation test, not the final baseline model.

In [ ]:
if False:

    import sys
    from ultralytics import YOLO

    sys.path.insert(0, "/content/road-damage-code")
    from src.train import prepare_data_yaml

    # Generate the correct dataset configuration
    data_yaml = prepare_data_yaml()

    # Load pretrained YOLO26s
    model = YOLO("yolo26s.pt")

    # Short test before full training
    model.train(
        data=str(data_yaml),
        epochs=1,
        fraction=0.01,
        imgsz=640,
        batch=8,
        device=0,
        workers=2,
        val=False,
        seed=42,
        project="/content/road-damage-code/runs",
        name="YOLO26s_smoke_test",
    )

The smoke test successfully verified model initialization, dataset loading, GPU training, validation, and checkpoint saving.


## Full-Dataset Training Benchmark

A one-epoch benchmark is performed using the complete training split
to estimate GPU memory usage and training time.

### Configuration
- Model: Pretrained YOLO26s
- Training images: 30,704 (100%)
- Epochs: 1
- Image size: 640 × 640
- Batch size: 8
- GPU: NVIDIA Tesla T4
- Seed: 42

This benchmark is not intended to produce a final model.

In [ ]:
if False:

    import time
    from ultralytics import YOLO
    from src.train import prepare_data_yaml

    data_yaml = prepare_data_yaml()

    model = YOLO("yolo26s.pt")

    start = time.time()

    model.train(
        data=str(data_yaml),
        epochs=1,
        fraction=1.0,
        imgsz=640,
        batch=8,
        device=0,
        workers=2,
        seed=42,
        val=False,
        plots=False,
        close_mosaic=0,
        project="/content/road-damage-code/runs",
        name="YOLO26s_full_dataset_benchmark",
    )

    elapsed = time.time() - start

    print(f"Total benchmark time: {elapsed / 60:.2f} minutes")

### Benchmark Results

- **Total runtime:** 15.89 minutes
- **GPU memory:** 2.43 GB

The benchmark completed successfully on the Tesla T4. A 50-epoch run may require approximately 11–14 hours or more, so checkpointing, training resumption and switching to a more powerful gpu will be important.

## YOLO26s Baseline Training
Train YOLO26s for 50 epochs using our existing training script.

###Configuration

- **Model:** Pretrained YOLO26s
- **Dataset:** RDD2022 (30,704 train / 3,835 val)
- **Epochs:** 50
- **Image size:** 640 × 640
- **Optimizer:** MuSGD (auto)
- **Augmentation:** Ultralytics defaults
- **Batch size:** 8
- **GPU:** NVIDIA L4
- **Seed:** 42


In [ ]:

# !python /content/road-damage-code/src/train.py

### YOLO26s Baseline Training Results

- **Training time:** 6.371 hours
- **Precision:** 65.2%
- **Recall:** 54.8%
- **mAP@50:** 59.7%
- **mAP@50–95:** 31.6%

The baseline training completed successfully. These results are from the validation set using `best.pt`. Final performance will be measured on the untouched test set.

### Baseline Model Analysis

- **Learning:** Training losses decreased steadily across 50 epochs.
- **Overfitting:** No major overfitting observed.
- **Performance on validation set**: began to plateau near the end.
- **Best class:** Alligator Crack (69.9% AP@50).
- **Weakest class:** Pothole (51.2% AP@50).
- **Class confusion:** Longitudinal and transverse cracks were sometimes confused.
- **Missed detections:** Some genuine road damages were missed, as seen in the Norwegian validation samples.

**Conclusion:** The baseline learned useful road-damage features but has room for improvement. Next, we investigate domain-wise performance and target specific weaknesses.

## Country-wise Validation Evaluation
Evaluate the trained YOLO26s model separately across all 7 RDD2022 domains.

In [ ]:
#!python /content/road-damage-code/src/evaluate_by_country.py

### Country-wise Evaluation Findings

- **Best domain:** China MotorBike (91.7% mAP@50).
- **Weakest domain:** Norway (26.9% mAP@50, 25.9% recall).
- **India:** 42.0% mAP@50.
- Performance varies significantly across the 7 RDD2022 domains.
- Differences may reflect camera viewpoints, road conditions, and class distributions.

**Next:** Investigate Norway and India to identify targeted improvements.

## Experiment 2 — YOLO26m (Larger Model)

Evaluate whether using a larger capacity model improves overall road-damage
detection performance compared with our YOLO26s baseline.



### YOLO26m Full-Dataset Training Benchmark

A one-epoch benchmark is performed on the complete training dataset
to estimate GPU memory usage and training time before full training.

#### Configuration
- Model: Pretrained YOLO26m
- Training images: 30,704 (100%)
- Epochs: 1
- Image size: 640 × 640
- Batch size: 8
- GPU: NVIDIA L4
- Seed: 42

This benchmark is only for measuring training feasibility and cost.
It is not intended to produce the final YOLO26m model.


In [ ]:
if False:

    import sys
    import time
    from ultralytics import YOLO

    sys.path.insert(0, "/content/road-damage-code")
    from src.train import prepare_data_yaml

    # Prepare the dataset configuration
    data_yaml = prepare_data_yaml()

    # Load the pretrained YOLO26m model
    model = YOLO("yolo26m.pt")

    # Run the 1-epoch training benchmark
    start_time = time.time()
    results = model.train(
        data=str(data_yaml),
        epochs=1,
        imgsz=640,
        batch=8,
        device=0,
        workers=2,
        val=False,
        plots=False,
        close_mosaic=0,
        seed=42,
        project="/content/road-damage-code/runs",
        name="YOLO26m_benchmark"
    )

    elapsed_time = time.time() - start_time
    print(f"YOLO26m benchmark training completed in: {elapsed_time / 60:.2f} minutes")


### YOLO26m Benchmark Results

- **Total runtime:** 11.34 minutes
- **GPU memory:** 4.65 GB

The benchmark completed successfully on the NVIDIA L4 GPU.
YOLO26m fits comfortably in GPU memory, making full training feasible.


### YOLO26m Full Training

Train YOLO26m for 50 epochs using our existing training script.

#### Configuration

- **Model:** Pretrained YOLO26m
- **Dataset:** RDD2022 (30,704 train / 3,835 val)
- **Epochs:** 50
- **Image size:** 640 × 640
- **Optimizer:** Auto
- **Augmentation:** Ultralytics defaults
- **Batch size:** 8
- **GPU:** NVIDIA L4
- **Seed:** 42

**Hypothesis:** Increasing model capacity may improve overall
road-damage detection performance compared with our YOLO26s baseline.

The trained model will be evaluated using precision, recall,
mAP@50, and mAP@50–95.


In [ ]:
#!python /content/road-damage-code/src/train_yolo26m.py


### YOLO26m Full Training Results


- **Precision:** 64.6%
- **Recall:** 55.7%
- **mAP@50:** 60.3%
- **mAP@50–95:** 31.9%

The YOLO26m training completed successfully.

Compared with the YOLO26s baseline, YOLO26m achieved only
a small improvement in overall validation mAP.

**Conclusion:** Increasing model capacity did not produce a
substantial accuracy improvement. YOLO26s remains the preferred
deployment candidate due to its smaller size and similar performance.

Both models will be retained for reproducibility and further comparison.
